# Intertemporal treatment in ModernDiD and R

This notebook is generated from the [Intertemporal treatment in ModernDiD and R](https://moderndid.readthedocs.io/en/stable/r_comparisons/r_intertemporal.html) page of the ModernDiD documentation.

This comparison checks whether [`did_multiplegt`](https://moderndid.readthedocs.io/en/stable/api/generated/didinter/moderndid.did_multiplegt.html) and R's
`DIDmultiplegtDYN::did_multiplegt_dyn` return the same effects for the
banking deregulation panel. Both estimate two post-treatment horizons and
one placebo without normalization or covariates and cluster analytical
standard errors by state. The output shows each fitted result before
comparing the estimates and standard errors directly.

R runs inside this Python session through `rpy2` and receives the same
data as the Python estimator. See the section on
[running the comparisons](https://moderndid.readthedocs.io/en/stable/r_comparisons/index.html#r-comparisons-running) for the setup
that runs these cells locally.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    # Colab ships rpy2 3.5 as a single package. Uninstalling it first keeps pip from deleting
    # the files it shares with the split rpy2 3.6 packages.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "rpy2"], check=True)
    # rpy2 supports Colab's R 4.6 from release 3.6.7 on.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "moderndid[all]==0.3.1", "rpy2>=3.6.7"], check=True)
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "r-cran-didmultiplegtdyn"], check=True)
    # R polars isn't on CRAN. With NOT_CRAN set, its installer downloads a prebuilt Rust library.
    subprocess.run(["Rscript", "-e", 'Sys.setenv(NOT_CRAN = "true"); install.packages("polars", repos = "https://rpolars.r-universe.dev"); if (!requireNamespace("polars", quietly = TRUE)) quit(status = 1)'], check=True)
else:
    print("Outside Colab, set up R as https://moderndid.readthedocs.io/en/stable/r_comparisons/index.html#r-comparisons-running describes.")

## The same county panel

[`load_favara_imbs`](https://moderndid.readthedocs.io/en/stable/api/generated/data/moderndid.load_favara_imbs.html) loads the county panel used in the
[intertemporal treatment example](https://moderndid.readthedocs.io/en/stable/user_guide/example_inter_did.html#example-inter-did).
`Dl_vloans_b` measures the change in log mortgage lending. `inter_bra`
counts the banking restrictions that a county's state had lifted in each
year and can change more than once. The 50 state codes in `state_n` define
the clusters in both calls.

We keep the 928 of 1,048 counties whose lending outcome is observed in all
12 years from 1994 through 2005. Since both estimators receive this complete
panel of 11,136 county-years, the comparison does not depend on how either
implementation handles incomplete county histories.

In [ ]:
import polars as pl
import rpy2.robjects as ro

import moderndid as did

data = did.load_favara_imbs().select("county", "year", "Dl_vloans_b", "inter_bra", "state_n")
data = data.drop_nulls().filter(pl.len().over("county") == 12)

In [ ]:
import numpy as np
import pandas as pd
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr

# DIDmultiplegtDYN 2.4.0 uses the attached R polars package.
with ro.default_converter.context():
    importr("polars")

with (ro.default_converter + pandas2ri.converter).context():
    ro.globalenv["lending"] = ro.conversion.py2rpy(data.to_pandas())

## Fit the same specification

Without normalization, each horizon describes the effect of the
treatment paths that switchers actually followed. The placebo uses
outcomes before their first treatment change. Both implementations
compare switchers with groups that had the same starting treatment and
had not yet changed it. Those comparison groups include counties whose
treatment never changes.

In [ ]:
python_fit = did.did_multiplegt(
    data,
    yname="Dl_vloans_b", tname="year", idname="county", dname="inter_bra",
    effects=2, placebo=1, normalized=False,
    xformla="~1", cluster="state_n", boot=False,
)
print(python_fit)

In [ ]:
with ro.default_converter.context():
    r_fit = ro.r("""
        DIDmultiplegtDYN::did_multiplegt_dyn(
            df = lending,
            outcome = "Dl_vloans_b", time = "year", group = "county", treatment = "inter_bra",
            effects = 2, placebo = 1, normalized = FALSE,
            controls = NULL, cluster = "state_n", graph_off = TRUE
        )
    """)
    ro.r["print"](r_fit.rx2("results").rx2("Effects"))
    ro.r["print"](r_fit.rx2("results").rx2("Placebos"))

## Compare estimates and uncertainty

The next table takes each estimate and standard error from its fitted
object. The differences subtract R from Python before any rounding.
Unlike a bootstrap comparison, these analytical standard errors have no
simulation draws to align.

In [ ]:
with ro.default_converter.context():
    r_effects = r_fit.rx2("results").rx2("Effects")
    r_placebos = r_fit.rx2("results").rx2("Placebos")
    r_estimates = np.concatenate([r_effects.rx(True, "Estimate"), r_placebos.rx(True, "Estimate")])
    r_se = np.concatenate([r_effects.rx(True, "SE"), r_placebos.rx(True, "SE")])
python_estimates = np.concatenate([python_fit.effects.estimates, python_fit.placebos.estimates])
python_se = np.concatenate([python_fit.effects.std_errors, python_fit.placebos.std_errors])

np.testing.assert_allclose(python_estimates, r_estimates, rtol=1e-5, atol=1e-6)
np.testing.assert_allclose(python_se, r_se, rtol=1e-3, atol=1e-6)
pd.DataFrame({
    "Quantity": ["Effect 1", "Effect 2", "Placebo 1"],
    "Python estimate": python_estimates,
    "R estimate": r_estimates,
    "Estimate difference": python_estimates - r_estimates,
    "Python SE": python_se,
    "R SE": r_se,
    "SE difference": python_se - r_se,
})

This comparison covers the two horizon estimates, the placebo, and their
state-clustered analytical standard errors. Changing normalization,
controls, or the comparison groups would define a different check.

## Versions used

These versions of Python, R, and the packages produced the output shown on
this page.

In [ ]:
from importlib.metadata import version
from platform import python_version

with ro.default_converter.context():
    r_version = ro.r("R.version.string")[0]
    did_version = ro.r('as.character(packageVersion("DIDmultiplegtDYN"))')[0]
pd.DataFrame({
    "component": ["Python", "ModernDiD", "rpy2", "R", "DIDmultiplegtDYN"],
    "version": [python_version(), version("moderndid"), version("rpy2"), r_version, did_version],
})